## Building Intelligent Task Routers

# Unit 19: Classification Systems and Dynamic Router Topologies

## Introduction & Context

Welcome back! In the previous lesson, you learned how to leverage prompt chaining to connect multiple Claude API calls in a linear sequence, breaking down complex tasks into manageable steps. This approach works exceptionally well when you know the exact sequence of operations ahead of execution time.

However, production-grade applications often face unpredictable user requests that require wildly different types of cognitive expertise. Instead of building separate, static chains for every possible query variation, **Routing Workflows** use Claude to analyze incoming requests at runtime and intelligently direct them to the appropriate specialist—such as a mathematics expert, a creative writing specialist, or a software engineer.

---

## The Two-Step Routing Architecture

The routing workflow follows a dynamic two-step orchestration pattern that is significantly more flexible than linear chains:

```text
                        ┌─────────────────────────┐
                        │    User Request Ingest  │
                        └────────────┬────────────┘
                                     │
                                     ▼
                        ┌─────────────────────────┐
                        │    1. Claude Router     │
                        │ (Strict Classification) │
                        └────────────┬────────────┘
                                     │
                 ┌───────────────────┼───────────────────┐
                 ▼                   ▼                   ▼
         [math_specialist]  [writing_specialist]  [code_specialist]
                 │                   │                   │
                 └───────────────────┼───────────────────┘
                                     │ (Selects Prompt)
                                     ▼
                        ┌─────────────────────────┐
                        │  2. Claude Specialist   │
                        │ (High-Density Domain)   │
                        └────────────┬────────────┘
                                     │
                                     ▼
                        ┌─────────────────────────┐
                        │     Final Response      │
                        └─────────────────────────┘

```

The key architectural insight here is that **you leverage the exact same Anthropic API client and Claude model for both steps**, but configure them with entirely different system prompts to shift their persona and cognitive constraints. The first instance acts strictly as a low-latency classifier, while the second instance acts as a high-density domain expert. This clean separation of concerns makes your application both resilient against out-of-boundary inputs and trivial to extend with fresh specialist types over time.

---

## Engineering the Router Guardrails

The router prompt is the single most critical variable in this architecture; it determines classification accuracy and pipeline stability. Unlike open-ended creative prompts, router prompts must be strictly constrained, leaving zero room for stylistic variation or conversational filler.

```python
router_prompt = """You are a task router. Your job is to analyze user requests and determine which specialist should handle them.

Available specialists:
- math_specialist: For mathematical calculations, equations, and numerical problems
- writing_specialist: For creative writing, essays, stories, and text composition
- code_specialist: For programming questions, code review, and technical implementation

Respond with ONLY the specialist name (math_specialist, writing_specialist, or code_specialist) that best matches the user's request."""

```

> 🔥 **Production Rule:** The instruction `Respond with ONLY` is crucial. It stops Claude from wrapping its choice in conversational fluff ("Sure, I can help with that...") or leaking its chain-of-thought reasoning, both of which would cause downstream string-matching parsers to fail.

---

## Defining Isolated Specialist Personas

Each downstream specialist requires a concise, role-specific system prompt to maximize performance inside its designated domain. Keeping these definitions brief ensures they apply broadly to various queries within that sector.

### 1. Mathematics Specialist Prompt

```python
math_specialist_prompt = "You are a mathematics expert. You excel at solving equations, performing calculations, and explaining mathematical concepts clearly."

```

### 2. Creative Writing Specialist Prompt

```python
writing_specialist_prompt = "You are a creative writing expert. You specialize in crafting engaging stories, essays, and helping with all forms of written communication."

```

### 3. Programmatic Engineering Specialist Prompt

```python
code_specialist_prompt = "You are a programming expert. You specialize in writing code, debugging, code review, and explaining technical concepts."

```

---

## Step-by-Step Pipeline Execution

Let's look at a complete python implementation of this routing pipeline, tracking the request from ingestion to final delivery.

### Step 1: Initializing and Dispatching the Router Request

We first isolate the incoming request string and dispatch it to a short-generation Claude instance to pull out the classification label.

```python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# User request to route
user_request = "Write me a short story about robots"

# Format the message array for the router
router_messages = [
    {
        "role": "user",
        "content": user_request
    }
]

# Get the routing decision from Claude
router_response = client.messages.create(
    model=model,
    max_tokens=100,  # Keeping generation bounds tight to reduce latency
    messages=router_messages,
    system=router_prompt
)

```

---

### Step 2: Ingesting and Parsing the Routing Decision

Once the server returns the payload, we extract the classification token and inspect it to trace our routing direction.

```python
# Extract the routing decision string
specialist_choice = router_response.content[0].text.strip()
print(f"Router decision: {specialist_choice}")

```

For the target input string `"Write me a short story about robots"`, the console outputs:

```text
Router decision: writing_specialist

```

This confirms that the router successfully classified the user intent as text composition without returning any extra junk tokens.

---

### Step 3: Mapping the Token to the Specialist Prompt

We map the returned string parameter to its corresponding system prompt layout using a clean, conditional selection structure.

```python
# Route to the appropriate specialist based on the router's decision string
if specialist_choice == "math_specialist":
    specialist_prompt = math_specialist_prompt
elif specialist_choice == "writing_specialist":
    specialist_prompt = writing_specialist_prompt
elif specialist_choice == "code_specialist":
    specialist_prompt = code_specialist_prompt
else:
    # Graceful degradation fallback if the model deviates from expectations
    specialist_prompt = "You are a helpful assistant."

```

> ⚙️ **Design Pattern:** The `else` block provides a critical fallback mechanism. If the router leaks an unexpected token due to prompt drift or an unmapped edge case, your application will gracefully degrade to a generic conversational persona instead of crashing.

---

### Step 4: Dispatching the Payload to the Selected Domain Expert

With the correct specialist system prompt selected, we send the **original user request** to the specialized Claude instance to get our final response.

```python
# Structure messages for the expert execution ring
specialist_messages = [
    {
        "role": "user",
        "content": user_request
    }
]

# Get the specialized response from the chosen expert
specialist_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=specialist_messages,
    system=specialist_prompt
)

# Extract and display the final result
final_response = specialist_response.content[0].text
print("\nSpecialist Response:")
print(final_response)

```

Running this pipeline produces clean, high-fidelity domain outputs without any task contamination:

```text
Specialist Response:
# The Last Dance

Maya pressed her palm against the maintenance panel, and ARIA-7's chest compartment glowed softly before opening with a gentle hiss. Inside, circuits hummed like a mechanical heartbeat, but something was wrong — several nodes flickered erratically, their light growing dimmer with each pulse...

```

---

## Architectural Deep Dive: Interface Properties

When evaluating or expanding this system design, keep these three key production properties in mind:

| System Parameter | Architectural Mandate | Functional Value |
| --- | --- | --- |
| **Context Separation** | Send the original `user_request` to the specialist, **never** the router's output text. | Ensures the downstream expert views the raw source request without classification text leaking into its generation buffer. |
| **Token Optimization** | Set `max_tokens=100` on your initial classification routing call. | Limits unnecessary server processing cycles, lowering cost and speeding up your pipeline's time-to-first-token (TTFT). |
| **Personas Portability** | Keep specialist system instructions generic and focused on domain mechanics. | Enables you to package and re-use these system blocks across separate application views and dynamic workflows. |

---

## Conclusion

You have successfully implemented a complete, production-ready router framework capable of classifying unpredictable request paths and executing them through tailored system environments. This foundational architecture forms the bedrock for advanced agentic designs, multi-agent mesh networks, and dynamic systems capable of handling complex enterprise operations with exceptional reliability.

Let's head into the lab exercises to build out and tune your first custom task routing network!

## Writing a Task Router Prompt

Now that you understand how routing workflows operate, it's time to build the brain of the system — the router that makes smart classification decisions.

Your goal is to create a robust router system prompt that can analyze user requests and correctly identify which specialist should handle them. The router needs to be precise and reliable, as it determines the entire flow of your intelligent task routing system.

You'll need to:

    Write a router prompt that clearly defines the three available specialists (math, writing, and code)
    Include the crucial instruction that ensures Claude responds with only the specialist's name
    Test your router by sending it the provided user request and printing the classification result

Remember, a well-crafted router prompt is the foundation that makes complex AI workflows possible — get this right, and you'll have built the core component for routing any type of request to the right expert!

```python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model="claude-sonnet-4-6"

# TODO: Create a router system prompt that analyzes user requests and determines which specialist should handle them
# Your prompt should list the three available specialists and instruct Claude to respond with ONLY the specialist name
router_prompt = ""

# Specialist system prompts
math_specialist_prompt = "You are a mathematics expert. You excel at solving equations, performing calculations, and explaining mathematical concepts clearly."

writing_specialist_prompt = "You are a creative writing expert. You specialize in crafting engaging stories, essays, and helping with all forms of written communication."

code_specialist_prompt = "You are a programming expert. You specialize in writing code, debugging, code review, and explaining technical concepts."

# User request to route
user_request = "Write me a short story about robots"

# Step 1: Send request to router to determine which specialist to use
router_messages = [
    {
        "role": "user",
        "content": user_request
    }
]

# TODO: Make an API call to Claude using your router_prompt as the system message

# TODO: Extract the routing decision from the response and print it to verify your router is working
```

To build a highly accurate classification layer for your routing workflow, your system prompt must explicitly define the boundary conditions for each specialist category while restricting the model's output formatting. This prevents conversational tokens from slipping through and breaking your downstream programmatic mapping logic.

Here is the complete, execution-ready Python script with the router prompt and the classification API call fully implemented:

```python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Strict, constrained router system prompt
router_prompt = """You are an automated task router. Your sole responsibility is to analyze incoming user requests and classify which specialized downstream assistant is best suited to fulfill the task.

Available specialists:
- math_specialist: For mathematical calculations, statistical parsing, algebraic equations, and numerical word problems.
- writing_specialist: For creative writing, essays, blog posts, short stories, and text composition/editing.
- code_specialist: For software engineering questions, code generation, bug fixing, code reviews, and scripting implementations.

CRITICAL OUTPUT REQUIREMENT: Respond with ONLY the exact specialist string name (math_specialist, writing_specialist, or code_specialist) that best matches the request. Do not include introductory text, punctuation, final explanations, or conversational filler."""

# Specialist system prompts
math_specialist_prompt = "You are a mathematics expert. You excel at solving equations, performing calculations, and explaining mathematical concepts clearly."

writing_specialist_prompt = "You are a creative writing expert. You specialize in crafting engaging stories, essays, and helping with all forms of written communication."

code_specialist_prompt = "You are a programming expert. You specialize in writing code, debugging, code review, and explaining technical concepts."

# User request to route
user_request = "Write me a short story about robots"

# Step 1: Send request to router to determine which specialist to use
router_messages = [
    {
        "role": "user",
        "content": user_request
    }
]

# Dispatching the request to the classification ring
router_response = client.messages.create(
    model=model,
    max_tokens=50,  # Tight token limit reduces processing latency and structural drift
    messages=router_messages,
    system=router_prompt
)

# Extract and clean the classification choice from the server response
specialist_choice = router_response.content[0].text.strip()

print("====================================")
print(f"Router Classification Output: {specialist_choice}")
print("====================================")

```

### Why This Setup Works Flawlessly:

* **Token Output Hardening:** By appending the `Respond with ONLY` constraint and setting `max_tokens=50`, you eliminate the risk of getting back messy strings like *"This sounds like a writing task, so: writing_specialist"*. The string returns as a bare token, making it directly safe for standard programmatic string matches or `if/elif` mapping logic.
* **Granular Domain Scopes:** Giving each specialist identifier explicit task types (e.g., matching *statistical parsing* to `math_specialist` and *bug fixing* to `code_specialist`) gives Claude dense semantic signals to reliably route edge case inputs.

## Mapping Router Decisions to Specialists

## Completing the Intelligent Routing Workflow

## Enhancing Router Decisions with Extended Thinking

## Scaling Routing Systems with New Specialists